In [ ]:
import math
import warnings

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
import matplotlib.pyplot as plt
import numpy as np
import sympy as sym

from rotation.symbolic.general import ANGLE, MASS, AXIAL_COUPLING, PARAMETERS, create_3d_rotation
from rotation.symbolic.axial import create_kinetic, create_axial_mix_equation
from rotation.numerical.solver import MASS_NEUTRAL_PION, MASS_ETA, MASS_ETA_PRIME, initialize_system, create_func, solve_mixing
from rotation.numerical.axial import calculate_uvw, axial_calculate_bare_masses, axial_calculate_meson_couplings, axial_calculate_bare_kaon, axial_calculate_kaon_couplings

In [ ]:
particles = tuple(["pi0", "eta", "etaP"])

In [ ]:
def format_output(key, value):
    """Helps format solutions for printing"""
    if key in [MASS["pi0"], MASS["pi0Bare"]]:
        value *= sym.Symbol(r" m_{\pi^0\text{, PDG}}") / MASS_NEUTRAL_PION
    elif key in [MASS["eta"], MASS["etaBare"]]:
        value *= sym.Symbol(r" m_{\eta\text{, PDG}}") / MASS_ETA
    elif key in [MASS["etaP"], MASS["etaPBare"]]:
        value *= sym.Symbol(r" m_{\eta^{\prime}\text{, PDG}}") / MASS_ETA_PRIME
    elif key in [MASS["AP"], MASS["pi0-AP"], MASS["eta-AP"], MASS["etaP-AP"]]:
        value *= sym.Symbol(r" \text{ GeV}")
    elif key in ANGLE.values():
        value *= sym.Symbol(r" \pi") / math.pi
    return value

## Prepare solver

In [ ]:
rotate = create_3d_rotation()[0]
kin = create_kinetic()
equation = create_axial_mix_equation(rotate, kin)[0]
argument_order, system_of_equations = initialize_system(equation.lhs - equation.rhs)

## Solver inputs

In [ ]:
mA = 1
quarks = 0.7, 0.1, -0.2
piA, etaA, etaPA = axial_calculate_bare_masses(
    coupling_up=quarks[0],
    coupling_down=quarks[1],
    coupling_strange=quarks[2],
)
kin_u, kin_v, kin_w = calculate_uvw(
    m_axial=mA,
    m_pi0_axial=piA,
    m_eta_axial=etaA,
    m_etap_axial=etaPA,
)
# Fixed known parameters
known = {
    MASS["pi0"]: MASS_NEUTRAL_PION,
    MASS["eta"]: MASS_ETA,
    MASS["etaP"]: MASS_ETA_PRIME,
    ANGLE["u"]: kin_u,
    ANGLE["v"]: kin_v,
    ANGLE["w"]: kin_w,
}

# These will be the unknowns we numerically solve for (the values are
# just initial guesses to get the solver going)
# Remember: the system of equations can only solve for up to 6 unknowns!
unknown = {
    MASS["pi0Bare"]: MASS_NEUTRAL_PION,
    MASS["etaBare"]: MASS_ETA,
    MASS["etaPBare"]: MASS_ETA_PRIME,
    ANGLE["alpha"]: 0.0,
    ANGLE["beta"]: 0.0,
    ANGLE["gamma"]: 0.0,
}

# Printout the system with our known values, and unknowns blank
print("Arguments input:")
for k in argument_order:
    if k in known:
        sym.Eq(k, known[k], evaluate=False)
    else:
        sym.Eq(k, sym.Symbol("?"), evaluate=False)
equation.subs(known)

## Run solver

In [ ]:
# Create function
func = create_func(
    system_of_equations,
    argument_order,
    known,
)
try:
    # Solve system
    solved = solve_mixing(
        func,
        argument_order,
        unknown,
    )
    print("Solver succeeded!")
except ValueError as e:
    print(e)
    print("The solver failed! Try other inputs.")

## Solution display

In [ ]:
# Printout the system with our solved unknowns, and knowns blank
print("Arguments solved:")
for k, v in solved.items():
    v = format_output(k, v)
    sym.Eq(k, v, evaluate=False)

print("The solved equation:")
equation.subs(solved)

print("Compare LHS and RHS evaluations:")
sym.Eq(
    (1*equation.lhs.subs(known|solved)).evalf(),
    (1*equation.rhs.subs(known|solved)).evalf(),
    evaluate=False
)

## Calculate Couplings

In [ ]:
lambA = 0.9
meson_couplings = axial_calculate_meson_couplings(
    m_axial = mA,
    l_axial = lambA,
    m_pi0_axial = piA,
    m_eta_axial = etaA,
    m_etap_axial = etaPA,
    alpha = solved[ANGLE["alpha"]],
    beta = solved[ANGLE["beta"]],
    gamma = solved[ANGLE["gamma"]],
    u = kin_u,
    v = kin_v,
    w = kin_w,
)
for s,g in zip([AXIAL_COUPLING[f"{k}-chi"] for k in particles], meson_couplings):
    sym.Eq(s,g,evaluate=False)

In [ ]:
bare_kaon_couplings = axial_calculate_bare_kaon(
    energy_pion=1,
    energy_dm=1,
    m_axial=mA,
    m_pi0_axial=piA,
    m_eta_axial=etaA,
    m_etap_axial=etaPA,
    coupling_up=quarks[0],
    coupling_down=quarks[1],
    coupling_strange=quarks[2],
)
for s,g in zip([AXIAL_COUPLING[f"{k}Bare-pi-K"] for k in particles], bare_kaon_couplings):
    sym.Eq(s,g,evaluate=False)

print("After rotating:")
kaon_couplings = axial_calculate_kaon_couplings(
    *bare_kaon_couplings,
    u = kin_u,
    v = kin_v,
    w = kin_w,
    alpha = solved[ANGLE["alpha"]],
    beta = solved[ANGLE["beta"]],
    gamma = solved[ANGLE["gamma"]],
)
for s,g in zip([AXIAL_COUPLING[f"{k}-pi-K"] for k in particles], kaon_couplings):
    sym.Eq(s,g,evaluate=False)

In [ ]:
# Creating the kaon couplings as a function of the pion energy
def kaon_function(epi, edm):
    return axial_calculate_kaon_couplings(
        *axial_calculate_bare_kaon(
            energy_pion=epi,
            energy_dm=edm,
            m_axial=mA,
            m_pi0_axial=piA,
            m_eta_axial=etaA,
            m_etap_axial=etaPA,
            coupling_up=quarks[0],
            coupling_down=quarks[1],
            coupling_strange=quarks[2],
        ),
        u = kin_u,
        v = kin_v,
        w = kin_w,
        alpha = solved[ANGLE["alpha"]],
        beta = solved[ANGLE["beta"]],
        gamma = solved[ANGLE["gamma"]]
    )

dms = np.linspace(0,10,num=11)
pis = np.linspace(0,10,num=11)
fig, ax = plt.subplots(1,1)
ys = np.array([kaon_function(pis[10], x) for x in dms])
for s, y in zip([AXIAL_COUPLING[k] for k in ["pi0-pi-K", "eta-pi-K", "etaP-pi-K"]], ys.T):
    _ = ax.plot(dms, y, label=f"${s}$")
_ = ax.legend()

## Loop the Solver

In [ ]:
x_label = MASS["AP"]
x_min, x_max = 0, 5

warnings.filterwarnings("error", category=RuntimeWarning)
solutions = []
for x in np.linspace(x_min, x_max,  num=101):
    try:
        kinetics = calculate_uvw(x, piA, etaA, etaPA)
    except RuntimeWarning as e:
        print(f"{x = :.3e}", end="\t")
        print(f"UVW RuntimeWarning: {e}")
        continue
    func = create_func(
        system_of_equations,
        argument_order,
        known | {ANGLE[k]: v for k,v in zip(["u", "v", "w"], kinetics)},
    )
    try:
        solved = solve_mixing(
            func,
            argument_order,
            unknown,
        )
        solutions.append([x, known.copy(), solved.copy()])
    except ValueError as e:
        print(f"{x = :.3e}", end="\t")
        print(f"Solver ValueError: {e}")
    except RuntimeWarning as e:
        print(f"{x = :.3e}", end="\t")
        print(f"Solver RuntimeWarning: {e}")
warnings.resetwarnings()

In [ ]:
xs = np.array([sol[0] for sol in solutions])
# Reorganize solutions to be arrays of each unknown
data = {}
for sol in solutions:
    for y_label, data_point in sol[2].items():
        try:
            data[y_label].append(data_point)
        except KeyError:
            data[y_label] = [data_point]
for y_label, ys in data.items():
    data[y_label] = np.array(ys)
    data[y_label] = np.ma.array(ys, mask=data[y_label] < 0.0)

if not data:
    print("No data!")
else:
    # plotting
    fig, axs = plt.subplots(1,2, figsize=(10,6))
    mass_selector = {MASS[k]: v for k,v in zip(["pi0Bare", "etaBare", "etaPBare"], [MASS_NEUTRAL_PION, MASS_ETA, MASS_ETA_PRIME])}

    for y_label, ys in data.items():
        if y_label in ANGLE.values():
            ax = axs[1]
            y_data = ys.data
        else:
            ax = axs[0]
            y_data = ys.data / mass_selector[y_label]
        _ = ax.scatter(xs, y_data, label=f"${y_label}$", marker=".")
    _ = axs[0].set_ylabel("Bare Mass / Rotated Mass")
    _ = axs[1].set_ylabel("Angle")
    _ = fig.suptitle(", ".join([f"${PARAMETERS['a'+q]} = {quarks[i]}$" for i,q in enumerate("uds")]))
    for ax in axs.flatten():
        _ = ax.set_xlim(x_min, x_max)
        _ = ax.set_xlabel(f"${x_label}$")
        _ = ax.legend()
        _ = ax.grid(True)
    _ = fig.tight_layout()
    fig.show()